In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'Warehouse'))

In [ ]:
from Inventory_Builder import Inventory_Builder, InventoryConfig
from Warehouse_Builder import Warehouse_Builder, WarehouseConfig, AisleConfig
from Storage_Primitive import Storage_Type, Singleton, Pallet

## Parameters

- `num_skus` — number of cartons to generate
- `handling_splits` — probability weights for `['conveyable', 'non-conveyable']`
- `category_splits` — probability weights for `['food', 'clothing', 'electronic', 'furniture', 'seasonal', 'chemical']`

In [ ]:
_st = Storage_Type()
print('Handling types :', _st.handling_storage_types)
print('Category types :', _st.category_storage_types)
print('Unit types     :', _st.unit_storage_types)

In [ ]:
inventory_config = InventoryConfig(
    num_skus=50,
    handling_splits=[0.7, 0.3],
    category_splits=[0.2, 0.2, 0.2, 0.2, 0.1, 0.1],
)

warehouse_config = WarehouseConfig(
    total_aisles=4,
    aisle_splits=[0.5, 0.5],
    aisle_configs=[
        AisleConfig(handling_type='conveyable',     storage_type='food',      unit_type='pallet',    bayXPerAisle=4, bayYPerAisle=6, storage_sizes=['medium']),
        AisleConfig(handling_type='non-conveyable', storage_type='furniture', unit_type='singleton', bayXPerAisle=4, bayYPerAisle=6, storage_sizes=['large']),
    ]
)

## Build

In [ ]:
inventory = Inventory_Builder().from_config(inventory_config).build()
warehouse = Warehouse_Builder().from_config(warehouse_config).build()

print(f'SKUs      : {len(inventory.cartons)}')
print(f'Aisles    : {len(warehouse.aisles)}')
print(f'Total bins: {len(warehouse.bins)}')

## Carton Dimensions

In [ ]:
import pandas as pd

rows = [{
    'sku'        : c.sku,
    'handling'   : c.storage_type[0],
    'category'   : c.storage_type[1],
    'lift_group' : c.lift_group,
    'length'     : c.length,
    'width'      : c.width,
    'height'     : c.height,
    'weight'     : c.weight,
    'volume'     : c.volume(),
    'demand_rate': round(c.demand.rate, 2),
} for c in inventory.cartons]

df = pd.DataFrame(rows).set_index('sku')
df.head(10)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['handling'].value_counts().plot(kind='bar', ax=axes[0], rot=0)
axes[0].set_title('Handling type split')
axes[0].set_ylabel('SKU count')

df['category'].value_counts().plot(kind='bar', ax=axes[1], rot=30)
axes[1].set_title('Category type split')
axes[1].set_ylabel('SKU count')

plt.tight_layout()
plt.show()

## Dimension Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col in zip(axes, ['length', 'width', 'height']):
    ax.hist(df[col], bins=20, edgecolor='black')
    ax.set_title(f'{col.capitalize()} distribution')
    ax.set_xlabel('Units')
    ax.set_ylabel('SKU count')

plt.tight_layout()
plt.show()

## Demand Rate Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['demand_rate'], bins=30, edgecolor='black')
ax.set_title('Demand rate distribution (Poisson λ per SKU)')
ax.set_xlabel('Rate')
ax.set_ylabel('SKU count')
plt.tight_layout()
plt.show()

## Simulated Pick Quantities
Sample one pick quantity per SKU and inspect the resulting distribution.

In [ ]:
df['sampled_picks'] = [c.demand.sample() for c in inventory.cartons]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['sampled_picks'], bins=30, edgecolor='black')
ax.set_title('Simulated pick quantities across SKUs')
ax.set_xlabel('Pick quantity')
ax.set_ylabel('SKU count')
plt.tight_layout()
plt.show()

df['sampled_picks'].describe()